# Model Comparison

ELPD-Based Model Evaluation

We calculated the expected log pointwise predictive density (ELPD) for each model to summarize its expected out-of-sample predictive performance. Differences in ELPD should be interpreted alongside their standard errors, which quantify uncertainty in the estimated predictive performance.

In [3]:
from cmdstanpy import CmdStanModel
import arviz as az
import pickle as pkl
import os
import xarray as xr
import numpy as np

In [4]:
PROJECT_ROOT = '/home/apoorva/Desktop/MZB-new-analysis/'
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

Working directory: /home/apoorva/Desktop/MZB-new-analysis


In [5]:
#open the data file
with open('data/New output (pkl)/Steady State/SHM/SHM_T1.pkl', 'rb') as f:
    fit_1 = pkl.load(f)

with open('data/New output (pkl)/Steady State/SHM/SHM_T2.pkl', 'rb') as f:
    fit_2 = pkl.load(f)

with open('data/New output (pkl)/Steady State/SHM/SHM_FO_SPLINE4.pkl', 'rb') as f:
    fit_3 = pkl.load(f)

with open('data/New output (pkl)/Steady State/SHM/SHM_T2MZP.pkl', 'rb') as f:
    fit_4 = pkl.load(f)
    
# with open('/home/apoorva/Desktop/work/MZ_analysis/data/New output (pkl)/Steady State/TDM/TDM_division_T1.pkl', 'rb') as f:
#     fit_5 = pkl.load(f)

# with open('/home/apoorva/Desktop/work/MZ_analysis/data/New output (pkl)/Steady State/TDM/TDM_division_T2.pkl', 'rb') as f:
#     fit_6 = pkl.load(f)

# with open('/home/apoorva/Desktop/work/MZ_analysis/data/New output (pkl)/Steady State/TDM/TDM_division_FO.pkl', 'rb') as f:
#     fit_7 = pkl.load(f)
    
# with open('/home/apoorva/Desktop/work/MZ_analysis/data/New output (pkl)/Steady State/TDM/TDM_division_T2MZP.pkl', 'rb') as f:
#     fit_8 = pkl.load(f)

# with open('/home/apoorva/Desktop/work/MZ_analysis/data/New output (pkl)/Steady State/TDM/TDM_influx_T1.pkl', 'rb') as f:
#     fit_9 = pkl.load(f)

# with open('/home/apoorva/Desktop/work/MZ_analysis/data/New output (pkl)/Steady State/TDM/TDM_influx_T2.pkl', 'rb') as f:
#     fit_10 = pkl.load(f)

# with open('/home/apoorva/Desktop/work/MZ_analysis/data/New output (pkl)/Steady State/TDM/TDM_influx_FO.pkl', 'rb') as f:
#     fit_11 = pkl.load(f)

# with open('/home/apoorva/Desktop/work/MZ_analysis/data/New output (pkl)/Steady State/TDM/TDM_influx_T2MZP.pkl', 'rb') as f:
#     fit_12 = pkl.load(f)

# data_file = os.path.join('/home/apoorva/Desktop/work/MZ_analysis/data/data.json')

ModuleNotFoundError: No module named 'numpy._core'

In [ ]:
# Convert the fit objects to arviz InferenceData objects with log_likelihoods array
# Dictionary to store idata objects
idata_dict = {}

for i in range(1, 5):
	fit_var = eval(f"fit_{i}")  # Dynamically access fit_1, fit_2, ..., fit_22
	idata_dict[f"idata_{i}"] = az.from_cmdstanpy(fit_var, log_likelihood={"log_lik_Mz": "log_lik_Mz", "log_lik_Nfd": "log_lik_Nfd", "log_lik_Kid": "log_lik_Kid", "log_lik_Kih": "log_lik_Kih"})

# Dynamically assign idata_1, idata_2, ..., idata_22 for compatibility with later cells
globals().update(idata_dict)

In [ ]:
# Compute combined log-likelihood for model comparison
import xarray as xr

updated_idata_dict = {}

for i in range(1, 5):
    idata_i = eval(f"idata_{i}")
    
    # Sum all log-likelihood components
    combined_lik = (idata_i.log_likelihood["log_lik_Mz"].values +
                    idata_i.log_likelihood["log_lik_Nfd"].values +
                    idata_i.log_likelihood["log_lik_Kid"].values +
                    idata_i.log_likelihood["log_lik_Kih"].values)
    
    # Update log_likelihood with combined likelihood
    updated_log_likelihood = idata_i.log_likelihood.copy()
    updated_log_likelihood["combined_log_lik"] = xr.DataArray(
        combined_lik,
        dims=idata_i.log_likelihood["log_lik_Mz"].dims,
        coords=idata_i.log_likelihood["log_lik_Mz"].coords
    )
    
    updated_idata_dict[f"idata_{i}"] = az.InferenceData(
        posterior=idata_i.posterior,
        log_likelihood=updated_log_likelihood,
        sample_stats=idata_i.sample_stats
    )

globals().update(updated_idata_dict)

In [ ]:
# Create a dictionary of models dynamically
model_dict = {f"model_{i}": eval(f"idata_{i}") for i in range(1, 5)}  # Adjust range as needed

# Compare all models using LOO-CV
comparison_combined = az.compare(model_dict, var_name="combined_log_lik", method='BB-pseudo-BMA')

# Print the comparison results
print(comparison_combined)

/Users/apoorvasingh/Desktop/MZB-new-analysis/.venv/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/Users/apoorvasingh/Desktop/MZB-new-analysis/.venv/lib/python3.12/site-packages/arviz/stats/diagnostics.py:655: RuntimeWarning: invalid value encountered in scalar subtract
  if (np.max(ary) - np.min(ary)) < np.finfo(float).resolution:  # pylint: disable=no-member
/Users/apoorvasingh/Desktop/MZB-new-analysis/.venv/lib/python3.12/site-packages/arviz/stats/stats_utils.py:39: RuntimeWarning: invalid value encountered in subtract
  ary = ary - ary.mean(axis, keepdims=True)
/Users/apoorvasingh/Desktop/M

         rank   elpd_loo      p_loo  elpd_diff    weight         se       dse  \
model_2     0  38.197635  10.325277   0.000000  0.502462   9.973997  0.000000   
model_1     1  38.172372  10.453453   0.025262  0.495482  10.024498  0.606236   
model_3     2  25.450562  11.013745  12.747073  0.000414  10.919721  3.935955   
model_4     3  23.157373  10.995675  15.040261  0.001641  11.123423  4.599158   

         warning scale  
model_2     True   log  
model_1     True   log  
model_3     True   log  
model_4     True   log  


/Users/apoorvasingh/Desktop/MZB-new-analysis/.venv/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
